In [1]:
import argparse
import pandas as pd
import matplotlib
matplotlib.use('Agg')
from matplotlib import pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib import cm
import seaborn as sns

In [28]:
targets=['A1','A2','A3','A4','H1','H2','H3','H4']

for target in targets:
    print(target)
    file_name='../../raw_data/'+target+'/nanopolish/polya_results.tsv'
    tails=pd.read_csv(file_name,sep='\t')
    stats = tails.qc_tag.value_counts().to_frame()
    stats.to_csv('../../analysis/polya/'+target+'_all_summary.tsv', sep="\t", index=True) # report tsv
    pages = PdfPages('../../analysis/polya/'+target+'_all_summary.pdf')
    stats.plot(kind='barh', title="Nanopolish QC stats", fontsize=5)
    pages.savefig()
    plt.clf()
    pages.close()

A1
A2
A3
A4
H1
H2
H3
H4


In [3]:
stats

,qc_tag
PASS,379942
ADAPTER,154181
READ_FAILED_LOAD,43234
SUFFCLIP,40301
NOREGION,6510


In [8]:
tails

,readname,contig,position,leader_start,adapter_start,polya_start,transcript_start,read_rate,polya_length,qc_tag
0,442f691b-197c-4daf-a04c-39648ab692a9,AIPGENE6,900,2.0,3.0,5037.0,7889.0,86.06,76.46,PASS
1,5900629e-319e-4f95-8530-b096061d842f,AIPGENE6,311,2.0,3.0,14703.0,15545.0,97.16,22.13,ADAPTER
2,41cddc0a-cf94-4660-a5c2-8b5c3d3beb36,AIPGENE6,1096,2.0,3.0,11481.0,15151.0,107.57,126.04,ADAPTER
3,2148f375-fae5-4ce0-9042-7238d4504fca,AIPGENE6,33,2.0,3.0,15806.0,20119.0,83.67,114.78,ADAPTER
4,8f71807b-dcb5-4efb-89b4-1bee9bce0ba2,AIPGENE6,659,2.0,3.0,8890.0,10422.0,103.86,47.79,ADAPTER
...,...,...,...,...,...,...,...,...,...,...
624163,2eac8d19-3150-45bf-ae9e-f7c685b3e1c7,AIPGENE29261,3351,3.0,4.0,7955.0,8800.0,130.96,31.70,ADAPTER
624164,ead51b8e-46f2-40c0-9be0-89b3a4a92c74,AIPGENE29265,0,2.0,3.0,5165.0,6012.0,111.56,26.33,PASS
624165,4599bf77-e699-4d27-a3e5-40ac83bec2cb,AIPGENE29264,0,2.0,3.0,4602.0,9254.0,107.57,161.11,PASS
624166,3cdb3a67-33a6-487f-9a29-ce9fdb4ad02d,AIPGENE29262,328,309.0,676.0,7812.0,15787.0,103.86,269.97,PASS


# Draw Overview report of poly(A) tail lengths

In [6]:
import argparse
import numpy as np
import scipy
import pandas as pd
from collections import OrderedDict
import matplotlib
from Bio import SeqIO
matplotlib.use('Agg')
from matplotlib import pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib import cm
import seaborn as sns

In [35]:
def _make_distplot(data, title, label, xlab, ylab, pages):
    """ Make distplot with median. """
    ax = sns.distplot(data, kde=False, hist_kws={"label": label}, norm_hist=False)
    ax.set_title(title)
    ax.set_xlabel(xlab)
    ax.set_xlabel(xlab)
    ax.legend(loc='best')
    pages.savefig()
    plt.clf()


def _make_boxplot(df, med, title,pages):
    """ Make boxplot. """
    ax = sns.boxplot(x="polya_length", data=df, showfliers=False, orient='v')
    ax.set_title(title)
    ax.text(0, med + 0.5, np.round(med, 2), horizontalalignment='center', size='x-small', color='w', weight='semibold')
    pages.savefig()
    plt.clf()


In [38]:
sns.set_style("whitegrid")

for target in targets:
    print(target)
    file_name='../../raw_data/'+target+'/nanopolish/polya_results_pass_only.tsv'
    tails=pd.read_csv(file_name,sep='\t',header=None)
    header_name={0:'readname',1:'contig',2:'position',3:'leader_start',4:'adapter_start',5:'polya_start',6:'transcript_start',7:'read_rate',8:'polya_length',9:'qc_tag'}
    tails=tails.rename(columns=header_name)
    mdf = tails[['contig', 'polya_length']].groupby(['contig']).polya_length.agg(['median', 'count']).reset_index()
    mdf = mdf.rename(columns={"median": "polya_length"})
    mdf.sort_values(by="count", ascending=False, inplace=True)
    mdf.to_csv('../../analysis/polya/'+target+'_median_polya_length.csv', sep="\t", index=False)
    pages = PdfPages('../../analysis/polya/'+target+'_polya_length_summary.pdf')
    _make_distplot(tails.polya_length.values, title="Global tail length distribution.", label="Median: {:.2f}".format(tails.polya_length.median()), xlab="Tail length", ylab="Count", pages=pages)
    _make_boxplot(tails, med=tails.polya_length.median(), title="Global tail length distibution (no outliers).", pages=pages)

    '''
    for tr in mdf.contig.values:
        med = mdf[mdf.contig == tr].polya_length.values[0]
        _make_distplot(tails[tails.contig == tr].polya_length, title="Tail length distribution: {}".format(tr), label="Median: {:.2f}".format(med), xlab="Tail length", ylab="Count", pages=pages)
        _make_boxplot(tails[tails.contig == tr], med=med, title="Tail length distibution (no outliers): {}".format(tr), pages=pages)
    '''
    pages.close()

A1


/tmp/ipykernel_172576/3626675955.py:3: UserWarning: 

`distplot` is a deprecated function and will be removed in seaborn v0.14.0.

Please adapt your code to use either `displot` (a figure-level function with
similar flexibility) or `histplot` (an axes-level function for histograms).

For a guide to updating your code to use the new functions, please see
https://gist.github.com/mwaskom/de44147ed2974457ad6372750bbe5751

  ax = sns.distplot(data, kde=False, hist_kws={"label": label}, norm_hist=False)
/home/zhonh0b/miniconda3/envs/epigenetic/lib/python3.8/site-packages/seaborn/_oldcore.py:1599: UserWarning: Vertical orientation ignored with only `x` specified.
  warnings.warn(single_var_warning.format("Vertical", "x"))


A2


/tmp/ipykernel_172576/3626675955.py:3: UserWarning: 

`distplot` is a deprecated function and will be removed in seaborn v0.14.0.

Please adapt your code to use either `displot` (a figure-level function with
similar flexibility) or `histplot` (an axes-level function for histograms).

For a guide to updating your code to use the new functions, please see
https://gist.github.com/mwaskom/de44147ed2974457ad6372750bbe5751

  ax = sns.distplot(data, kde=False, hist_kws={"label": label}, norm_hist=False)
/home/zhonh0b/miniconda3/envs/epigenetic/lib/python3.8/site-packages/seaborn/_oldcore.py:1599: UserWarning: Vertical orientation ignored with only `x` specified.
  warnings.warn(single_var_warning.format("Vertical", "x"))


A3


...

/tmp/ipykernel_172576/3626675955.py:3: UserWarning: 

`distplot` is a deprecated function and will be removed in seaborn v0.14.0.

Please adapt your code to use either `displot` (a figure-level function with
similar flexibility) or `histplot` (an axes-level function for histograms).

For a guide to updating your code to use the new functions, please see
https://gist.github.com/mwaskom/de44147ed2974457ad6372750bbe5751

  ax = sns.distplot(data, kde=False, hist_kws={"label": label}, norm_hist=False)
/home/zhonh0b/miniconda3/envs/epigenetic/lib/python3.8/site-packages/seaborn/_oldcore.py:1599: UserWarning: Vertical orientation ignored with only `x` specified.
  warnings.warn(single_var_warning.format("Vertical", "x"))


H3


/tmp/ipykernel_172576/3626675955.py:3: UserWarning: 

`distplot` is a deprecated function and will be removed in seaborn v0.14.0.

Please adapt your code to use either `displot` (a figure-level function with
similar flexibility) or `histplot` (an axes-level function for histograms).

For a guide to updating your code to use the new functions, please see
https://gist.github.com/mwaskom/de44147ed2974457ad6372750bbe5751

  ax = sns.distplot(data, kde=False, hist_kws={"label": label}, norm_hist=False)
/home/zhonh0b/miniconda3/envs/epigenetic/lib/python3.8/site-packages/seaborn/_oldcore.py:1599: UserWarning: Vertical orientation ignored with only `x` specified.
  warnings.warn(single_var_warning.format("Vertical", "x"))


H4


/tmp/ipykernel_172576/3626675955.py:3: UserWarning: 

`distplot` is a deprecated function and will be removed in seaborn v0.14.0.

Please adapt your code to use either `displot` (a figure-level function with
similar flexibility) or `histplot` (an axes-level function for histograms).

For a guide to updating your code to use the new functions, please see
https://gist.github.com/mwaskom/de44147ed2974457ad6372750bbe5751

  ax = sns.distplot(data, kde=False, hist_kws={"label": label}, norm_hist=False)
/home/zhonh0b/miniconda3/envs/epigenetic/lib/python3.8/site-packages/seaborn/_oldcore.py:1599: UserWarning: Vertical orientation ignored with only `x` specified.
  warnings.warn(single_var_warning.format("Vertical", "x"))


In [19]:

tails = pd.read_csv('../../raw_data/A4/nanopolish/polya_results_pass_only.tsv', sep="\t",header=None)
tails

,0,1,2,3,4,5,6,7,8,9
0,442f691b-197c-4daf-a04c-39648ab692a9,AIPGENE6,900,2.0,3.0,5037.0,7889.0,86.06,76.46,PASS
1,d47d8969-d269-453b-869a-4727c7c206fe,AIPGENE7,1319,2.0,3.0,4484.0,6426.0,120.48,72.64,PASS
2,a2562520-fb47-4b5c-a9f7-f15b0a37dc48,AIPGENE6,1196,2.0,3.0,8353.0,9781.0,107.57,45.96,PASS
3,04c7e3b2-055d-4b54-8537-9fd33fbe00d6,AIPGENE7,923,87.0,89.0,6766.0,9051.0,111.56,79.59,PASS
4,36f6c1ea-008f-4946-a50d-8b0e61fa204e,AIPGENE6,436,2.0,3.0,6130.0,7521.0,136.91,58.18,PASS
...,...,...,...,...,...,...,...,...,...,...
379937,0251af2a-40f9-424a-bf13-0702f2b7cb17,AIPGENE29267,0,2.0,3.0,4817.0,9464.0,120.48,180.84,PASS
379938,bfcbca67-7250-44ca-a6a8-3af647984a25,AIPGENE29262,286,2.0,3.0,6031.0,9715.0,115.85,136.65,PASS
379939,ead51b8e-46f2-40c0-9be0-89b3a4a92c74,AIPGENE29265,0,2.0,3.0,5165.0,6012.0,111.56,26.33,PASS
379940,4599bf77-e699-4d27-a3e5-40ac83bec2cb,AIPGENE29264,0,2.0,3.0,4602.0,9254.0,107.57,161.11,PASS


In [21]:
tails

,readname,contig,position,leader_start,adapter_start,polya_start,transcript_start,read_rate,polya_length,qc_tag
0,442f691b-197c-4daf-a04c-39648ab692a9,AIPGENE6,900,2.0,3.0,5037.0,7889.0,86.06,76.46,PASS
1,d47d8969-d269-453b-869a-4727c7c206fe,AIPGENE7,1319,2.0,3.0,4484.0,6426.0,120.48,72.64,PASS
2,a2562520-fb47-4b5c-a9f7-f15b0a37dc48,AIPGENE6,1196,2.0,3.0,8353.0,9781.0,107.57,45.96,PASS
3,04c7e3b2-055d-4b54-8537-9fd33fbe00d6,AIPGENE7,923,87.0,89.0,6766.0,9051.0,111.56,79.59,PASS
4,36f6c1ea-008f-4946-a50d-8b0e61fa204e,AIPGENE6,436,2.0,3.0,6130.0,7521.0,136.91,58.18,PASS
...,...,...,...,...,...,...,...,...,...,...
379937,0251af2a-40f9-424a-bf13-0702f2b7cb17,AIPGENE29267,0,2.0,3.0,4817.0,9464.0,120.48,180.84,PASS
379938,bfcbca67-7250-44ca-a6a8-3af647984a25,AIPGENE29262,286,2.0,3.0,6031.0,9715.0,115.85,136.65,PASS
379939,ead51b8e-46f2-40c0-9be0-89b3a4a92c74,AIPGENE29265,0,2.0,3.0,5165.0,6012.0,111.56,26.33,PASS
379940,4599bf77-e699-4d27-a3e5-40ac83bec2cb,AIPGENE29264,0,2.0,3.0,4602.0,9254.0,107.57,161.11,PASS


In [22]:
mdf = tails[['contig', 'polya_length']].groupby(['contig']).polya_length.agg(['median', 'count']).reset_index()
mdf = mdf.rename(columns={"median": "polya_length"})
mdf.sort_values(by="count", ascending=False, inplace=True)
mdf

,contig,polya_length,count
5455,AIPGENE20026,63.530,8041
10566,AIPGENE3167,63.685,7476
5591,AIPGENE20312,65.300,3552
13576,AIPGENE865,55.140,3385
1436,AIPGENE12775,65.190,2946
...,...,...,...
3894,AIPGENE17098,193.600,1
12383,AIPGENE6361,95.230,1
3875,AIPGENE17069,120.620,1
12386,AIPGENE6370,119.690,1


In [29]:
med = mdf[mdf.contig == 'AIPGENE20026'].polya_length.values[0]

In [30]:
med

63.53